In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
import numpy as np

from src.utils import (
    get_args,
    set_seed,
    get_datesets_and_loaders,
    get_trained_VAE,
    get_trained_VAE_with_domain_classifier,
    get_trained_classifier,
    get_trained_classifier_Base,
    test_model,
    prepare_report,
    run_all_senario
)

/home/asad/workspace/DomainProject/changeDomain/notebooks/effective-gzsda/gzsda/src/utils.py:3: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.1)
  import scipy


In [3]:

DOMAIN_SET = ['angry', 'childlike', 'depressed', 'neutral', 'old', 'proud', 'strutting']
DATA_DIR = './data/ActionStyleDataset/'
DATASET_DETAILS = {
    'prefix': 'ActionStyle-',
    'suffix': '-clip.mat',
    'resnet_feature': 'clip_features',
    'split_file_name': 'instanceSplit_actionStyle_unseen2.mat',
}
NUM_LABELS = 5

In [4]:
import sys

sys.argv.extend([
    "--encoder_layer_sizes", "512", "512",
    "--decoder_layer_sizes", "512", "512",
])

In [5]:
import json
from pathlib import Path

RESULT_OBJ_PATH = "./result/json/actionStyle.json"
RESULT_CSV_PATH = "./result/csv/actionStyle.csv"
path = Path(RESULT_OBJ_PATH)

if path.exists():
    with path.open("r", encoding="utf-8") as f:
        result = json.load(f)
else:
    result = {}

result.keys()

dict_keys(['base', 'CCVAE', 'our0', 'our_GRE'])

In [6]:
base = "base"
CCVAE = "CCVAE"
our0 = "our0"
our_GRE = "our_GRE"

# clear last result
# result.pop(base, None)
# result.pop(CCVAE, None)
# result.pop(our0, None)
# result.pop(our_GRE, None)

## Base

In [7]:
def main_base(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    classifier = get_trained_classifier_Base(
        data_loaders=data_loaders,
        NUM_LABELS=NUM_LABELS,
        device=device,
        input_dim=512)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [8]:
if base not in result:
    result[base] = run_all_senario(main_base, DOMAIN_SET, input_dim=512, num_trial=6)

# GZSDA

In [9]:
def main_gzsda(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    vae = get_trained_VAE(
        data_loaders=data_loaders,
        args=args,
        device=device)

    classifier = get_trained_classifier(
        data_loaders=data_loaders,
        vae=vae,
        NUM_LABELS=NUM_LABELS,
        device=device,
        input_dim=512)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [10]:
if CCVAE not in result:
    result[CCVAE] = run_all_senario(main_gzsda, DOMAIN_SET, input_dim=512, num_trial=6)

## m0

In [11]:
def main_m0(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    vae = get_trained_VAE(
        data_loaders=data_loaders,
        args=args,
        device=device)

    classifier = get_trained_classifier(
        data_loaders=data_loaders,
        vae=vae,
        NUM_LABELS=NUM_LABELS,
        device=device,
        change_policy_epoch=30,
        input_dim=512)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [12]:
if our0 not in result:
    result[our0] = run_all_senario(main_m0, DOMAIN_SET, input_dim=512, num_trial=6)

## m1: seperate after encoder

In [13]:
def main_m1(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    vae = get_trained_VAE_with_domain_classifier(
        data_loaders=data_loaders,
        args=args,
        device=device)
        
    classifier = get_trained_classifier(
        data_loaders=data_loaders,
        vae=vae,
        NUM_LABELS=NUM_LABELS,
        device=device,
        change_policy_epoch=30,
        input_dim=512)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [14]:
if our_GRE not in result:
    result[our_GRE] = run_all_senario(main_m1, DOMAIN_SET, input_dim=512, num_trial=6)


## Merge results

In [15]:
with open(RESULT_OBJ_PATH, "w") as f:
    json.dump(result, f, indent=2)

In [16]:
# ignore our0
result.pop(our0, None)

{'angry -> childlike': 'Seen:     86.81 ± 5.12\nUnseen:   46.74 ± 12.14\nH-mean:   54.27 ± 11.76',
 'angry -> depressed': 'Seen:     86.65 ± 6.62\nUnseen:   44.67 ± 11.54\nH-mean:   52.74 ± 11.71',
 'angry -> neutral': 'Seen:     94.44 ± 5.56\nUnseen:   47.58 ± 12.26\nH-mean:   57.57 ± 12.90',
 'angry -> old': 'Seen:     92.43 ± 3.58\nUnseen:   45.25 ± 12.04\nH-mean:   54.54 ± 12.16',
 'angry -> proud': 'Seen:     98.81 ± 1.19\nUnseen:   46.27 ± 12.34\nH-mean:   57.75 ± 12.98',
 'angry -> strutting': 'Seen:     84.17 ± 8.21\nUnseen:   49.65 ± 12.91\nH-mean:   56.08 ± 13.22',
 'childlike -> angry': 'Seen:     97.92 ± 2.08\nUnseen:   36.87 ± 9.30\nH-mean:   49.35 ± 11.72',
 'childlike -> depressed': 'Seen:     94.15 ± 2.55\nUnseen:   32.63 ± 13.45\nH-mean:   39.74 ± 14.85',
 'childlike -> neutral': 'Seen:     97.16 ± 2.07\nUnseen:   40.39 ± 13.50\nH-mean:   49.60 ± 15.55',
 'childlike -> old': 'Seen:     95.30 ± 3.05\nUnseen:   13.98 ± 9.14\nH-mean:   19.98 ± 11.35',
 'childlike -> proud

In [17]:
import pandas as pd
import re

rows = [(k, m, result[m][k]) for m in result for k in result[m]]
df = pd.DataFrame(rows, columns=['domain', 'method', 'values'])

def extract_metrics(text):
    matches = dict(re.findall(r'(\w+):\s+([\d.]+\s*±\s*[\d.]+)', text))
    return pd.Series(matches)

df[['seen', 'unseen', 'H-mean']] = df['values'].apply(extract_metrics)
df = df[['domain', 'method', 'seen', 'unseen', 'H-mean']]

df['method'] = pd.Categorical(df['method'], categories=[base, CCVAE, our0, our_GRE], ordered=True)
df = df.sort_values(['domain', 'method']).reset_index(drop=True)

df

,domain,method,seen,unseen,H-mean
0,angry -> childlike,base,93.40 ± 2.82,30.35 ± 11.40,38.87 ± 12.93
1,angry -> childlike,CCVAE,90.62 ± 4.19,26.72 ± 11.77,32.82 ± 14.12
2,angry -> childlike,our_GRE,85.76 ± 6.49,48.19 ± 12.44,54.82 ± 11.97
3,angry -> depressed,base,94.08 ± 3.07,15.81 ± 4.59,25.60 ± 6.87
4,angry -> depressed,CCVAE,91.99 ± 4.95,8.41 ± 3.33,14.53 ± 5.40
...,...,...,...,...,...
121,strutting -> old,CCVAE,66.88 ± 8.50,10.62 ± 5.12,15.06 ± 6.92
122,strutting -> old,our_GRE,60.42 ± 8.18,30.41 ± 13.33,30.37 ± 10.54
123,strutting -> proud,base,75.00 ± 11.18,37.50 ± 13.31,39.63 ± 9.24
124,strutting -> proud,CCVAE,66.67 ± 10.54,28.95 ± 15.51,27.52 ± 12.62


In [18]:
df.to_csv(RESULT_CSV_PATH)

In [19]:
df.groupby("domain")

In [20]:
df["H-mean_value"] = (
    df["unseen"]
    # df["H-mean"]
    .str.split("±")
    .str[0]
    .astype(float)
)
df

,domain,method,seen,unseen,H-mean,H-mean_value
0,angry -> childlike,base,93.40 ± 2.82,30.35 ± 11.40,38.87 ± 12.93,30.35
1,angry -> childlike,CCVAE,90.62 ± 4.19,26.72 ± 11.77,32.82 ± 14.12,26.72
2,angry -> childlike,our_GRE,85.76 ± 6.49,48.19 ± 12.44,54.82 ± 11.97,48.19
3,angry -> depressed,base,94.08 ± 3.07,15.81 ± 4.59,25.60 ± 6.87,15.81
4,angry -> depressed,CCVAE,91.99 ± 4.95,8.41 ± 3.33,14.53 ± 5.40,8.41
...,...,...,...,...,...,...
121,strutting -> old,CCVAE,66.88 ± 8.50,10.62 ± 5.12,15.06 ± 6.92,10.62
122,strutting -> old,our_GRE,60.42 ± 8.18,30.41 ± 13.33,30.37 ± 10.54,30.41
123,strutting -> proud,base,75.00 ± 11.18,37.50 ± 13.31,39.63 ± 9.24,37.50
124,strutting -> proud,CCVAE,66.67 ± 10.54,28.95 ± 15.51,27.52 ± 12.62,28.95


In [21]:
best = df.loc[df.groupby("domain")["H-mean_value"].idxmax()]
best = best[["domain", "method", "H-mean"]].reset_index(drop=True)

best

,domain,method,H-mean
0,angry -> childlike,our_GRE,54.82 ± 11.97
1,angry -> depressed,our_GRE,53.49 ± 11.74
2,angry -> neutral,our_GRE,57.60 ± 12.90
3,angry -> old,our_GRE,53.40 ± 12.43
4,angry -> proud,our_GRE,58.23 ± 13.11
5,angry -> strutting,our_GRE,58.23 ± 13.19
6,childlike -> angry,our_GRE,46.10 ± 12.82
7,childlike -> depressed,our_GRE,45.69 ± 12.96
8,childlike -> neutral,our_GRE,52.13 ± 15.38
9,childlike -> old,our_GRE,19.79 ± 12.38


In [22]:
from collections import Counter
Counter(best.method)

Counter({'our_GRE': 39, 'base': 3})

In [23]:
8 + 34

42